# SynthFace on Google Colab: 3D synthetic identities (approach A)

Renders many synthetic identities with **Blender 5.2.2 + MPFB 2.0.17** (same versions and assets as the dev
laptop, every download sha256-checked) on the Colab **GPU**. Same seed gives the same identities as on the laptop.

**Before you start**
1. `Runtime > Change runtime type > T4 GPU` (or better).
2. GitLab: create a **read-only** token (Project or Personal access token, scope `read_repository`).
   In Colab click the key icon (Secrets) and add it as `GITLAB_TOKEN`; enable notebook access.
   The token is never printed or saved inside the clone.
3. Run the cells top to bottom. Output goes to Google Drive (`DRIVE_DIR`) so a disconnect loses nothing;
   re-running the render cell resumes where it stopped.

Rules (CLAUDE.md): license-clean assets only (all already in `assets/LICENSES.csv`), no AI-generated faces here,
never commit `output/`.

## 1. Settings

In [ ]:
JOB_ID = "colab_20x10"          # new name for every new batch
SEED = 20261013                 # same seed + same settings = same identities
NUM_IDENTITIES = 20
IMAGES_PER_IDENTITY = 10
SAMPLES = 48                    # Cycles samples (+ denoiser)
DEVICE = "OPTIX"                # "OPTIX" (T4/L4/A100), "CUDA" if OptiX fails, "CPU" without GPU
REPO = "gitlab.com/ankitscode/synthetic-image-genrator.git"
BRANCH = "master"               # master = 3D pipeline
DRIVE_DIR = "/content/drive/MyDrive/synthface_output"   # None = keep output only in this VM

PROJECT = "/content/synthface"
TOOLS = "/content/synthface_tools"
BLENDER = f"{TOOLS}/blender-5.2.2-linux-x64/blender"

## 2. Check the GPU

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > T4 GPU (or set DEVICE = 'CPU', very slow)"

## 3. Mount Drive and clone the repo (token from Colab Secrets)

In [ ]:
import os, subprocess
from google.colab import userdata

if DRIVE_DIR:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)

if not os.path.isdir(PROJECT):
    token = userdata.get("GITLAB_TOKEN")
    r = subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                        f"https://oauth2:{token}@{REPO}", PROJECT], capture_output=True, text=True)
    if r.returncode != 0:  # do not print the command: it contains the token
        raise RuntimeError("git clone failed: " + r.stderr.replace(token, "***"))
    subprocess.run(["git", "-C", PROJECT, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
print(subprocess.run(["git", "-C", PROJECT, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 4. Install Blender, MPFB, asset packs and HDRIs (about 3-5 min)

In [ ]:
!bash {PROJECT}/colab/setup_colab.sh {TOOLS}

## 5. Send output to Drive (optional)

In [ ]:
out = f"{PROJECT}/output"
if DRIVE_DIR and not os.path.islink(out):
    if os.path.isdir(out) and os.listdir(out):
        raise SystemExit(f"{out} already has files; move them first")
    if os.path.isdir(out):
        os.rmdir(out)
    os.symlink(DRIVE_DIR, out)
os.makedirs(out, exist_ok=True)
print(out, "->", os.path.realpath(out))

## 6. Render the identities

Prints one line per finished identity. Safe to re-run after a disconnect (`--resume` skips finished identities).
Blender's log is in `output/<JOB_ID>/blender_stderr.log`.

In [ ]:
import json, time

job_dir = f"{PROJECT}/output/{JOB_ID}"
os.makedirs(job_dir, exist_ok=True)
job = {"job_id": JOB_ID, "seed": SEED, "num_identities": NUM_IDENTITIES,
       "images_per_identity": IMAGES_PER_IDENTITY, "render": {"samples": SAMPLES, "device": DEVICE}}
job_path = f"{job_dir}/job.json"
if os.path.exists(job_path):
    if json.load(open(job_path)) != job:
        raise SystemExit(f"{job_path} exists with other settings: use a new JOB_ID")
else:
    json.dump(job, open(job_path, "w"), indent=2)

cmd = [BLENDER, "--background", "--python", "blender/render_batch.py", "--", "--job", job_path, "--resume"]
t0 = time.time()
with open(f"{job_dir}/blender_stderr.log", "a") as err:
    proc = subprocess.Popen(cmd, cwd=PROJECT, stdout=subprocess.PIPE, stderr=err, text=True)
    for line in proc.stdout:
        if line.startswith('{"event"'):
            ev = json.loads(line)
            note = " (already done)" if ev.get("skipped") else ""
            print(f"identity {ev['index']}/{NUM_IDENTITIES} {ev['status']}{note}  {time.time() - t0:.0f}s", flush=True)
print("Blender exit code:", proc.wait())

## 7. Contact sheet (one row per identity)

In [ ]:
!cd {PROJECT} && python tools/contact_sheet.py output/{JOB_ID} --max-cols 10
from IPython.display import Image, display
display(Image(f"{job_dir}/contact_sheet.png"))

## 8. Bring the batch back to the laptop

- With Drive: it is already in `MyDrive/synthface_output/<JOB_ID>/`. Download that folder.
- Without Drive: run the cell below to download a zip.

On the laptop, unzip to `output/<JOB_ID>/` in the project, then review with
`venv/bin/python tools/contact_sheet.py output/<JOB_ID>`.

In [ ]:
import shutil
from google.colab import files
zip_path = shutil.make_archive(f"/content/{JOB_ID}", "zip", job_dir)
files.download(zip_path)